In [1]:
import cv2
import numpy as np
from mediapipe import Image, ImageFormat
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("✓ Libraries imported successfully")
print(f"  OpenCV version: {cv2.__version__}")
print(f"  NumPy version: {np.__version__}")
print("✓ MediaPipe ready for hand detection")

✓ Libraries imported successfully
  OpenCV version: 4.8.1
  NumPy version: 1.26.4
✓ MediaPipe ready for hand detection


In [3]:
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Initialize the hand landmarker model
model_path = "hand_landmarker.task"

if not os.path.exists(model_path):
    print(f"❌ Error: {model_path} not found in current directory")
    print(f"   Current directory: {os.getcwd()}")
else:
    BaseOptions = python.BaseOptions
    HandLandmarker = vision.HandLandmarker
    HandLandmarkerOptions = vision.HandLandmarkerOptions
    RunningMode = vision.RunningMode
    
    # Create the hand landmarker
    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=RunningMode.IMAGE,
        num_hands=2  # Detect up to 2 hands
    )
    
    hand_landmarker = HandLandmarker.create_from_options(options)
    print("✓ Hand landmarker initialized successfully")
    print(f"  Model: {model_path}")
    print("  Ready to detect up to 2 hands")

✓ Hand landmarker initialized successfully
  Model: hand_landmarker.task
  Ready to detect up to 2 hands


Hand Landmarker Initialization Purpose

This module initializes the MediaPipe Hand Landmarker model that will be used to detect and track hand landmarks from webcam frames.

In [4]:
import cv2
import numpy as np
from mediapipe import Image, ImageFormat
import time
from collections import deque

# Configuration
MAX_FRAMES = 500  # Process up to 500 frames
DRAW_THICKNESS = 8  # Increased for better visibility
LANDMARK_RADIUS = 4
FINGER_RADIUS = 8
FRAME_DELAY = 0.05  # Delay between frames in seconds (50ms = ~20 FPS)
SMOOTH_BUFFER_SIZE = 3  # Buffer for smoothing coordinates

def is_index_finger_extended(hand_landmarks):
    """
    Check if index finger is extended (pointing).
    Index finger landmarks: 5=MCP, 6=PIP, 7=DIP, 8=TIP
    """
    # Get distances between consecutive finger joints
    tip = hand_landmarks[8]
    dip = hand_landmarks[7]
    pip = hand_landmarks[6]
    mcp = hand_landmarks[5]
    
    # Check if finger is extended (tip is higher/further than base)
    # Calculate vertical distance from MCP to TIP
    vertical_distance = abs(tip.y - mcp.y)
    
    # Finger is extended if TIP is significantly ahead of MCP
    return vertical_distance > 0.03  # Threshold for extension

def smooth_coordinates(x, y, history):
    """Smooth coordinates using historical data"""
    history.append((x, y))
    if len(history) > SMOOTH_BUFFER_SIZE:
        history.popleft()
    
    avg_x = int(np.mean([p[0] for p in history]))
    avg_y = int(np.mean([p[1] for p in history]))
    return avg_x, avg_y

def process_air_canvas(max_frames=MAX_FRAMES, frame_delay=FRAME_DELAY, show_display=True):
    """
    Process webcam input for air canvas drawing using ONLY the index finger.
    
    Args:
        max_frames: Maximum number of frames to process
        frame_delay: Delay between frames in seconds (slows down drawing)
        show_display: Whether to display the output window
    
    Returns:
        Dictionary with statistics about the session
    """
    try:
        cap = cv2.VideoCapture(0)
        
        if not cap.isOpened():
            print("❌ Error: Could not open webcam")
            return {"status": "failed", "reason": "webcam_not_available"}
        
        print("✓ Webcam opened successfully")
        print(f"Processing up to {max_frames} frames...")
        print(f"Frame delay: {frame_delay*1000:.0f}ms (~{1/frame_delay:.1f} FPS)")
        print(f"Line thickness: {DRAW_THICKNESS}px")
        print("📍 DRAWING: INDEX FINGER ONLY (NO EXTRA DOTS)\n")
        
        canvas = None
        prev_x, prev_y = 0, 0
        frame_count = 0
        hands_detected = 0
        lines_drawn = 0
        frames_no_hand = 0  # Track consecutive frames without hand
        coord_history = deque()  # History for smoothing
        is_drawing = False  # Track if currently drawing
        
        start_time = time.time()
        
        while frame_count < max_frames:
            success, frame = cap.read()
            
            if not success:
                print("Failed to read frame")
                break
            
            frame = cv2.flip(frame, 1)
            
            if canvas is None:
                canvas = np.zeros_like(frame)
            
            # Convert frame to MediaPipe Image format
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = Image(image_format=ImageFormat.SRGB, data=frame_rgb)
            
            # Detect hand landmarks
            detection_result = hand_landmarker.detect(mp_image)
            
            if detection_result.hand_landmarks and len(detection_result.hand_landmarks) > 0:
                hands_detected += 1
                frames_no_hand = 0  # Reset counter
                h, w, c = frame.shape
                
                # Use only the first hand for drawing
                hand_landmarks = detection_result.hand_landmarks[0]
                
                # ===== ONLY DRAW FROM INDEX FINGER (landmark 8) =====
                # Get index finger tip position
                x = int(hand_landmarks[8].x * w)
                y = int(hand_landmarks[8].y * h)
                
                # Check if index finger is extended
                index_extended = is_index_finger_extended(hand_landmarks)
                
                # Smooth the coordinates
                x, y = smooth_coordinates(x, y, coord_history)
                
                # Draw ONLY index finger tip indicator (no other landmarks)
                if index_extended:
                    # Bright green when extended (ready to draw)
                    cv2.circle(frame, (x, y), FINGER_RADIUS, (0, 255, 0), -1)
                    cv2.circle(frame, (x, y), FINGER_RADIUS + 2, (0, 255, 0), 2)  # Outline only
                    is_drawing = True
                    
                    # Only draw if we have a previous point and finger is extended
                    if prev_x > 0 and prev_y > 0:
                        cv2.line(canvas, (prev_x, prev_y), (x, y), (0, 255, 255), DRAW_THICKNESS)
                        lines_drawn += 1
                else:
                    # Red when folded (not drawing)
                    cv2.circle(frame, (x, y), FINGER_RADIUS, (0, 0, 255), -1)
                    is_drawing = False
                    prev_x, prev_y = 0, 0  # Reset on finger fold
                    coord_history.clear()
                
                # Always update position if extended
                if index_extended:
                    prev_x, prev_y = x, y
                
            else:
                frames_no_hand += 1
                # Only reset if we've had no hand for 5+ consecutive frames
                if frames_no_hand > 5:
                    prev_x, prev_y = 0, 0
                    coord_history.clear()
                    is_drawing = False
            
            output = cv2.add(frame, canvas)
            
            # Add status text with better formatting
            cv2.putText(output, "Air Canvas - INDEX FINGER ONLY", 
                       (10, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
            status_text = "✓ DRAWING" if is_drawing else "○ STAND BY"
            status_color = (0, 255, 0) if is_drawing else (200, 200, 0)
            cv2.putText(output, status_text, 
                       (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)
            
            cv2.putText(output, f"Frame: {frame_count + 1}/{max_frames} | Lines: {lines_drawn}", 
                       (10, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
            
            cv2.putText(output, "Extend index finger to draw | Fold to pause", 
                       (10, 120), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 100), 1)
            
            cv2.putText(output, "Press ESC to exit", 
                       (10, 140), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (100, 100, 100), 1)
            
            if show_display:
                cv2.imshow("Air Canvas", output)
                key = cv2.waitKey(1) & 0xFF
                if key == 27:  # ESC key
                    print(f"\nEarly exit at frame {frame_count + 1}")
                    break
            
            frame_count += 1
            
            # Add delay to slow down processing
            time.sleep(frame_delay)
        
        elapsed_time = time.time() - start_time
        cap.release()
        cv2.destroyAllWindows()
        
        # Return statistics
        stats = {
            "status": "completed",
            "frames_processed": frame_count,
            "frames_with_hands": hands_detected,
            "lines_drawn": lines_drawn,
            "elapsed_time": f"{elapsed_time:.2f}s",
            "fps": f"{frame_count / elapsed_time:.1f}" if elapsed_time > 0 else "0"
        }
        
        print("\n✓ Air Canvas session completed!")
        print(f"  Frames processed: {stats['frames_processed']}")
        print(f"  Frames with hands: {stats['frames_with_hands']}")
        print(f"  Lines drawn: {stats['lines_drawn']}")
        print(f"  Time elapsed: {stats['elapsed_time']}")
        print(f"  FPS: {stats['fps']}")
        
        return stats
    
    except Exception as e:
        print(f"❌ Error: {e}")
        try:
            cap.release()
            cv2.destroyAllWindows()
        except:
            pass
        return {"status": "error", "error": str(e)}

# Run the air canvas
print("=" * 60)
print("Air Canvas Application - INDEX FINGER ONLY")
print("=" * 60)
print()

result = process_air_canvas(max_frames=MAX_FRAMES, frame_delay=FRAME_DELAY)

Air Canvas Application - INDEX FINGER ONLY

✓ Webcam opened successfully
Processing up to 500 frames...
Frame delay: 50ms (~20.0 FPS)
Line thickness: 8px
📍 DRAWING: INDEX FINGER ONLY (NO EXTRA DOTS)


✓ Air Canvas session completed!
  Frames processed: 500
  Frames with hands: 261
  Lines drawn: 239
  Time elapsed: 53.54s
  FPS: 9.3
